In [0]:
%sql
CREATE or replace TABLE orders (
    order_id INT,
    status VARCHAR(20),
    amount DECIMAL(10, 2)
);

INSERT INTO orders (order_id, status, amount) VALUES
    (101, 'completed', 150.00),
    (102, 'pending',   200.00),
    (103, 'completed',  90.00),
    (104, 'completed', 250.00),
    (105, 'cancelled', 180.00);


--1. Filter completed orders with amount above 100. Input: orders(order_id, status, amount)

select * from orders where amount >= 100 and status = 'completed'
 

In [0]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("FilterOrders").getOrCreate()

data = [
    (101, "completed", 150.00),
    (102, "pending",   200.00),
    (103, "completed",  90.00),
    (104, "completed", 250.00),
    (105, "cancelled", 180.00),
]

columns = ["order_id", "status", "amount"]

data_df = spark.createDataFrame(data,columns)

data_output_df = data_df.filter((F.col("amount") >= 100)  & (F.col("status") =='completed'))
display(data_output_df)

###1. Filter completed orders with amount above 100. Input: orders(order_id, status, amount)



In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import *

data = [
    ("S101", 100.00),
    ("S102", 250.00),
    ("S103", 75.50),
]
columns =["sale_id", "amount"]

data_df = spark.createDataFrame(data,columns)

#2.Add a tax column at 8% and a total-with-tax column. Input: sales(sale_id, amount).


data_tax_df = data_df.withColumn("tax" ,round(f.col("amount") * lit(0.08),2)).withColumn("total_with_tax",round(f.col("amount") + f.col("tax"),2)
)
display(data_tax_df)

In [0]:
%sql
CREATE or replace TABLE sales (
    sale_id VARCHAR(10),
    amount DECIMAL(10, 2)
);

INSERT INTO sales (sale_id, amount) VALUES
    ('S101', 100.00),
    ('S102', 250.00),
    ('S103', 75.50);

 --2.Add a tax column at 8% and a total-with-tax column. Input: sales(sale_id, amount).

 select sale_id, amount, round(amount * 0.08,2)  as tax ,round((amount  + amount *0.08),2) as total_with_tax from sales

In [0]:
from pyspark.sql import functions as F

data = [
    ("A", 2, 10.00),
    ("A", 3,  5.00),
    ("B", 1, 20.00),
    ("B", 4,  7.50),
    ("C", 2, 12.00),
]

sales_df = spark.createDataFrame(
    data,
    ["store_id", "quantity", "unit_price"]
)
#3. Calculate total sales by store. Input: sales(store_id, quantity, unit_price).

sales_output_df = (sales_df.groupBy(f.col("store_id")).agg(f.sum(f.col("quantity") * f.col("unit_price")).alias("total_sales")).orderBy("store_id")).select("store_id","total_sales")

display("sales_output_df")

In [0]:
%sql
CREATE or replace TABLE sales (
    store_id VARCHAR(10),
    quantity INT,
    unit_price DECIMAL(10, 2)
);

INSERT INTO sales (store_id, quantity, unit_price) VALUES
    ('A', 2, 10.00),
    ('A', 3,  5.00),
    ('B', 1, 20.00),
    ('B', 4,  7.50),
    ('C', 2, 12.00);

 
--3. Calculate total sales by store. Input: sales(store_id, quantity, unit_price).

select store_id , sum(quantity * unit_price) as total_sales
from sales
group by store_id
order by store_id

In [0]:
from pyspark.sql import functions as f

data = [
    ("2025-03-01 09:15:00", "click"),
    ("2025-03-01 10:20:00", "click"),
    ("2025-03-01 11:05:00", "purchase"),
    ("2025-03-02 08:30:00", "click"),
    (None, "click"),
    ("2025-03-02 12:10:00", None),
]

events_df = spark.createDataFrame(data, ["event_ts", "event_type"])

events_df = events_df.withColumn(
    "event_ts",
    F.to_timestamp("event_ts", "yyyy-MM-dd HH:mm:ss")
)

 
 
#4.Count events by calendar date and event type. Exclude null timestamps/types. Input: events(event_ts,event_type).

event_output_df =(events_df.filter(f.col("event_ts").isNotNull() & f.col("event_type").isNotNull())
                .groupBy(F.date_format("event_ts", "yyyy-MM-dd").alias("date"))
                .agg(F.count("*").alias("count"))
)
display(event_output_df)



In [0]:
%sql
CREATE or replace TABLE events (
    event_ts TIMESTAMP,
    event_type VARCHAR(50)
);

INSERT INTO events (event_ts, event_type) VALUES
    ('2025-03-01 09:15:00', 'click'),
    ('2025-03-01 10:20:00', 'click'),
    ('2025-03-01 11:05:00', 'purchase'),
    ('2025-03-02 08:30:00', 'click'),
    (NULL, 'click'),
    ('2025-03-02 12:10:00', NULL);

 
--4.Count events by calendar date and event type. Exclude null timestamps/types. Input: events(event_ts,event_type).

select date_format(event_ts,'yyyy-MM-dd') as event_ts,event_type,count(event_type)
from events
where event_ts is not null and event_type is not null
group by date_format(event_ts,'yyyy-MM-dd'),event_type
order by date_format(event_ts,'yyyy-MM-dd'),event_type

In [0]:
from pyspark.sql import functions as F

data = [
    ("Ana", "ana@example.com", "Boston"),
    ("Ben", "ben@example.com", None),
    ("Cara", None, "Seattle"),
    ("Dev", "dev@example.com", "Austin"),
]

employees_df = spark.createDataFrame(data, ["name", "email", "city"])

#Keep employees with a non-null email and replace null city values with "Unknown". Input:employees(name, email, city).
employee_output_df = employees_df.filter(f.col("name").isNotNull() & f.col("email").isNotNull()).fillna({"city":"unknown"})
display(employee_output_df)


In [0]:
%sql
CREATE or replace TABLE employees (
    name VARCHAR(100),
    email VARCHAR(255),
    city VARCHAR(100)
);

INSERT INTO employees (name, email, city) VALUES
    ('Ana', 'ana@example.com', 'Boston'),
    ('Ben', 'ben@example.com', NULL),
    ('Cara', NULL, 'Seattle'),
    ('Dev', 'dev@example.com', 'Austin');


--Keep employees with a non-null email and replace null city values with "Unknown". Input:employees(name, email, city).

select name,coalesce(city,'unknown') as city,email
from employees
where email is not null  

In [0]:
from pyspark.sql import functions as F

data = [
    ("C101", "O1"),
    ("C102", "O2"),
    ("C101", "O3"),
    ("C103", "O4"),
    ("C102", "O5"),
]

orders_df = spark.createDataFrame(data, ["customer_id", "order_id"])


#6. Return unique customer IDs. Input: orders(customer_id, order_id)

orders_output_df= orders_df.select("customer_id").distinct()
display(orders_output_df)

In [0]:
%sql
CREATE or replace TABLE orders (
    customer_id VARCHAR(20),
    order_id VARCHAR(20)
);

INSERT INTO orders (customer_id, order_id) VALUES
    ('C101', 'O1'),
    ('C102', 'O2'),
    ('C101', 'O3'),
    ('C103', 'O4'),
    ('C102', 'O5');


--6. Return unique customer IDs. Input: orders(customer_id, order_id)
select distinct customer_id
from orders
--group by customer_id
--having count(customer_id) = 1

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window

data = [
    ("Sales", 101, 75000),
    ("Sales", 102, 90000),
    ("Sales", 103, 90000),
    ("Engineering", 201, 110000),
    ("Engineering", 202, 105000),
    ("HR", 301, 65000),
]

employees_df = spark.createDataFrame(
    data,
    ["department", "employee_id", "salary"]
)


#Find the highest-paid employee in each department. Break ties by employee_id. Input:employees(department, employee_id, salary).

window = Window.partitionBy("department").orderBy(f.col("salary").desc(),f.col("employee_id").asc())

employee_output_df = employees_df.withColumn("rrn",f.row_number().over(window)).filter(f.col("rrn")<=2).drop(f.col("rrn"))
display(employee_output_df)


In [0]:
%sql
CREATE or replace TABLE employees (
    department VARCHAR(50),
    employee_id INT,
    salary DECIMAL(10, 2)
);

INSERT INTO employees (department, employee_id, salary) VALUES
    ('Sales', 101, 75000),
    ('Sales', 102, 90000),
    ('Sales', 103, 90000),
    ('Engineering', 201, 110000),
    ('Engineering', 202, 105000),
    ('HR', 301, 65000);


--Find the highest-paid employee in each department. Break ties by employee_id. Input:employees(department, employee_id, salary).
with ranked as (
    select employee_id,department,salary,row_number() over(partition by department order by salary desc ,employee_id asc) as rrn
    from employees
)
select employee_id,department,salary
from ranked
where rrn<=2

In [0]:
customers_data = [
    ("C1", "Asha"),
    ("C2", "Ravi"),
    ("C3", "Mei"),
    ("C4", "Omar"),
]

orders_data = [
    ("C1", "O101"),
    ("C3", "O102"),
    ("C1", "O103"),
]

customers_df = spark.createDataFrame(
    customers_data,
    ["customer_id", "name"]
)

orders_df = spark.createDataFrame(
    orders_data,
    ["customer_id", "order_id"]
)


without_orders_df = customers_df.alias("c").join(orders_df.alias("o"),(f.col("c.customer_id") == f.col("o.customer_id")),"left_anti")

display(without_orders_df)

#Find customers who have no orders. Input: customers(customer_id, name), orders(customer_id,order_id).

In [0]:
%sql
CREATE or replace TABLE customers (
    customer_id VARCHAR(10),
    name VARCHAR(100)
);

CREATE or replace TABLE orders (
    customer_id VARCHAR(10),
    order_id VARCHAR(10)
);

INSERT INTO customers (customer_id, name) VALUES
    ('C1', 'Asha'),
    ('C2', 'Ravi'),
    ('C3', 'Mei'),
    ('C4', 'Omar');

INSERT INTO orders (customer_id, order_id) VALUES
    ('C1', 'O101'),
    ('C3', 'O102'),
    ('C1', 'O103');



--Find customers who have no orders. Input: customers(customer_id, name), orders(customer_id,order_id).
select c.customer_id,c.name
from customers c left join orders o on c.customer_id = o.customer_id
where o.order_id is null


In [0]:
from pyspark.sql import functions as F

orders_data = [
    ("C1", "O101", 75.00),
    ("C2", "O102", 120.00),
    ("C1", "O103", 45.00),
]

customers_data = [
    ("C1", "Asha"),
    ("C2", "Ravi"),
]

orders_df = spark.createDataFrame(
    orders_data,
    ["customer_id", "order_id", "amount"]
)

customers_df = spark.createDataFrame(
    customers_data,
    ["customer_id", "name"]
)

#9. Return each order with the customer name. Input: orders(customer_id, order_id, amount),customers(customer_id, name).
customer_with_order_df = orders_df.alias("o").join(customers_df.alias("c"),(f.col("o.customer_id") == f.col("c.customer_id")),"left").select("c.customer_id","o.order_id","c.name","o.amount")
display(customer_with_order_df)

In [0]:
%sql
CREATE or replace TABLE orders (
    customer_id VARCHAR(10),
    order_id VARCHAR(10),
    amount DECIMAL(10, 2)
);

CREATE or replace TABLE customers (
    customer_id VARCHAR(10),
    name VARCHAR(100)
);

INSERT INTO orders (customer_id, order_id, amount) VALUES
    ('C1', 'O101', 75.00),
    ('C2', 'O102', 120.00),
    ('C1', 'O103', 45.00);

INSERT INTO customers (customer_id, name) VALUES
    ('C1', 'Asha'),
    ('C2', 'Ravi');


--9. Return each order with the customer name. Input: orders(customer_id, order_id, amount),customers(customer_id, name).

select c.customer_id,c.name,o.order_id,o.amount 
from orders o left join customers c on o.customer_id = c.customer_id
order by customer_id,order_id

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

tx_data = [
    ("A1", "2025-01-01 09:00:00", 101, 100.00),
    ("A1", "2025-01-01 09:05:00", 102, -25.00),
    ("A1", "2025-01-01 09:05:00", 103, 40.00),
    ("A2", "2025-01-01 10:00:00", 201, 200.00),
    ("A2", "2025-01-01 10:10:00", 202, -50.00),
]

tx_df = spark.createDataFrame(
    tx_data,
    ["account_id", "transaction_ts", "transaction_id", "amount"]
).withColumn(
    "transaction_ts",
    F.to_timestamp("transaction_ts", "yyyy-MM-dd HH:mm:ss")
)

#10. Calculate a running balance for each account ordered by timestamp and transaction ID. Input:tx(account_id, transaction_ts,transaction_id, amount).

running_balance_df = Window.partitionBy("account_id").orderBy("transaction_id","transaction_ts").rowsBetween(Window.unboundedPreceding,Window.currentRow)

running_balance_output_df = tx_df.withColumn("running_balance",f.sum(f.col("amount")).over(running_balance_df)).select("account_id","transaction_ts","transaction_id","amount","running_balance")
display(running_balance_output_df)

In [0]:
%sql
CREATE or replace TABLE tx (
    account_id VARCHAR(10),
    transaction_ts TIMESTAMP,
    transaction_id INT,
    amount DECIMAL(10, 2)
);

INSERT INTO tx (account_id, transaction_ts, transaction_id, amount) VALUES
    ('A1', '2025-01-01 09:00:00', 101, 100.00),
    ('A1', '2025-01-01 09:05:00', 102, -25.00),
    ('A1', '2025-01-01 09:05:00', 103, 40.00),
    ('A2', '2025-01-01 10:00:00', 201, 200.00),
    ('A2', '2025-01-01 10:10:00', 202, -50.00);

--10. Calculate a running balance for each account ordered by timestamp and transaction ID. Input:tx(account_id, transaction_ts, --transaction_id, amount).

select account_id,transaction_ts,transaction_id,amount,
sum(amount) over(partition by account_id order by transaction_ts,transaction_id rows between unbounded Preceding and current Row) as running_balance
from tx
order by account_id,transaction_ts,transaction_id
    


In [0]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window

products_data = [
    ("Books", 101, 30.00),
    ("Books", 102, 45.00),
    ("Books", 103, 45.00),
    ("Books", 104, 20.00),
    ("Electronics", 201, 500.00),
    ("Electronics", 202, 350.00),
    ("Electronics", 203, 600.00),
]

products_df = spark.createDataFrame(
    products_data,
    ["category", "product_id", "price"]
)
#11. Return the two highest-priced products in each category. Break ties by product_id. Input:products(category, product_id, price).

window = Window.partitionBy("category").orderBy(f.col("price").desc(),f.col("product_id"))
products_output_df = products_df.withColumn("rrn",f.row_number().over(window)).filter(f.col("rrn")<=2).drop(f.col("rrn")).orderBy("category","price","product_id")
display(products_output_df)

In [0]:
%sql
CREATE or replace TABLE products (
    category VARCHAR(50),
    product_id INT,
    price DECIMAL(10, 2)
);

INSERT INTO products (category, product_id, price) VALUES
    ('Books', 101, 30.00),
    ('Books', 102, 45.00),
    ('Books', 103, 45.00),
    ('Books', 104, 20.00),
    ('Electronics', 201, 500.00),
    ('Electronics', 202, 350.00),
    ('Electronics', 203, 600.00);



--11. Return the two highest-priced products in each category. Break ties by product_id. Input:

with ranked as (
    select *,row_number() over(partition by category order by price desc ,product_id asc) as rrn
    from products
)
select category,price,product_id
from ranked
where rrn<=2
order by category ,price,product_id


In [0]:
from pyspark.sql import functions as F

orders_data = [
    ("O101",),
    ("O102",),
    ("O101",),
    ("O103",),
    ("O102",),
    ("O101",),
]

orders_df = spark.createDataFrame(orders_data, ["order_id"])

#Find duplicate order IDs and their counts. Input: orders(order_id).

orders_output_df = orders_df.groupBy("order_id").count().filter(f.count("order_id")>1)
display(orders_output_df)


In [0]:
%sql
CREATE or replace TABLE orders (
    order_id VARCHAR(20)
);

INSERT INTO orders (order_id) VALUES
    ('O101'),
    ('O102'),
    ('O101'),
    ('O103'),
    ('O102'),
    ('O101');
--Find duplicate order IDs and their counts. Input: orders(order_id).

select order_id ,count(order_id) as count
from orders
group by order_id
HAVING count > 1; 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

customers_data = [
    ("C1", "2025-02-10 09:00:00", 101, "Asha"),
    ("C1", "2025-02-10 09:00:00", 102, "Asha P."),
    ("C1", "2025-02-08 14:30:00", 103, "Asha Old"),
    ("C2", "2025-02-12 11:15:00", 201, "Ravi"),
    ("C2", "2025-02-09 16:00:00", 202, "Ravi Old"),
]

customers_df = spark.createDataFrame(
    customers_data,
    ["customer_id", "updated_at", "record_id", "name"]
).withColumn(
    "updated_at",
    F.to_timestamp("updated_at", "yyyy-MM-dd HH:mm:ss")
)
#Keep the latest row per customer; break ties with the greatest record_id. Input:customers(customer_id, updated_at, record_id, name).
window = Window.partitionBy("customer_id").orderBy(f.col("updated_at").desc(),f.col("record_id").desc())
customer_output_df = customers_df.withColumn("rrn",f.row_number().over(window)).filter(f.col("rrn")== 1).drop(f.col("rrn"))
display(customer_output_df)


In [0]:
%sql
CREATE or replace TABLE customers (
    customer_id VARCHAR(20),
    updated_at TIMESTAMP,
    record_id INT,
    name VARCHAR(100)
);

INSERT INTO customers (customer_id, updated_at, record_id, name) VALUES
    ('C1', '2025-02-10 09:00:00', 101, 'Asha'),
    ('C1', '2025-02-10 09:00:00', 102, 'Asha P.'),
    ('C1', '2025-02-08 14:30:00', 103, 'Asha Old'),
    ('C2', '2025-02-12 11:15:00', 201, 'Ravi'),
    ('C2', '2025-02-09 16:00:00', 202, 'Ravi Old');

--Keep the latest row per customer; break ties with the greatest record_id. Input:customers(customer_id, updated_at, record_id, name).

with ranked as(
    select *,row_number() over (partition by customer_id order by to_timestamp(updated_at,'yyyy-MM-dd hh:MM:ss') desc,record_id desc) as rrn
    from customers
)

select customer_id,updated_at,record_id , name
from ranked
where rrn = 1

In [0]:
from pyspark.sql import functions as F
sales_data = [
    ("S1", "2025-01-05 10:15:00", 100.00),
    ("S1", "2025-01-20 14:30:00",  50.00),
    ("S1", "2025-02-03 09:00:00",  80.00),
    ("S2", "2025-01-12 16:45:00", 200.00),
    ("S2", "2025-02-18 11:20:00", 120.00),
]

sales_df = spark.createDataFrame(
    sales_data,
    ["store_id", "sale_ts", "amount"]
).withColumn(
    "sale_ts",
    F.to_timestamp("sale_ts", "yyyy-MM-dd HH:mm:ss")
)
#Calculate monthly revenue per store. Input: sales(store_id, sale_ts, amount).
sales_output_df = (
    sales_df
    .groupBy(
        "store_id",
        F.date_trunc("month", F.col("sale_ts")).alias("Month")
    )
    .agg(F.sum(f.col("amount")).alias("monthly_revenue"))
    .orderBy("store_id", "Month")
)
display(sales_output_df)

In [0]:
%sql
CREATE or replace TABLE sales (
    store_id VARCHAR(10),
.    sale_ts TIMESTAMP,
    amount DECIMAL(10, 2)
);

INSERT INTO sales (store_id, sale_ts, amount) VALUES
    ('S1', '2025-01-05 10:15:00', 100.00),
    ('S1', '2025-01-20 14:30:00',  50.00),
    ('S1', '2025-02-03 09:00:00',  80.00),
    ('S2', '2025-01-12 16:45:00', 200.00),
    ('S2', '2025-02-18 11:20:00', 120.00);

--Calculate monthly revenue per store. Input: sales(store_id, sale_ts, amount).

select store_id, sum(amount) as monthly_revenue, date_trunc('Month',sale_ts) as month
from sales
group by store_id,date_trunc('Month',sale_ts)
order by store_id,month    


In [0]:
from pyspark.sql import functions as F

items_data = [
    ("O101", "Notebook"),
    ("O101", "Pen"),
    ("O101", "Pencil"),
    ("O102", "Mug"),
    ("O102", "Tea"),
]

items_df = spark.createDataFrame(
    items_data,
    ["order_id", "product_name"]
)


#Collect product names into an array per order. Input: items(order_id, product_name).

products_per_order_df = (
    items_df.groupBy("order_id")
    .agg(f.collect_list(f.col("product_name")).alias("products"))
    .orderBy("order_id")
    
)
display(products_per_order_df)

In [0]:
%sql
CREATE or replace TABLE items (
    order_id VARCHAR(20),
    product_name VARCHAR(100)
);

INSERT INTO items (order_id, product_name) VALUES
    ('O101', 'Notebook'),
    ('O101', 'Pen'),
    ('O101', 'Pencil'),
    ('O102', 'Mug'),
    ('O102', 'Tea');


--Collect product names into an array per order. Input: items(order_id, product_name).

select order_id,collect_list("product_name") as products
from items
group by order_id
order by order_id


In [0]:
from pyspark.sql import functions as F

posts_data = [
    ("P1", ["spark", "data"]),
    ("P2", ["python", "sql", "analytics"]),
    ("P3", []),
]

posts_df = spark.createDataFrame(posts_data, ["post_id", "tags"])
#16. Explode an array of tags so there is one row per tag. Input: posts(post_id, tags), where tags is an --array

exploded_df = posts_df.withColumn("tags",f.explode("tags"))
display(exploded_df)

In [0]:
%sql
CREATE or replace TABLE posts (
    post_id VARCHAR(20),
    tags ARRAY<STRING>
);

INSERT INTO posts (post_id, tags) VALUES
    ('P1', ARRAY('spark', 'data')),
    ('P2', ARRAY('python', 'sql', 'analytics')),
    ('P3', ARRAY());

--16. Explode an array of tags so there is one row per tag. Input: posts(post_id, tags), where tags is an array.

select post_id,explode(tags)
from posts